# Camada Silver
---

In [0]:
# Criação do schema da camada Silver
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.silver")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

## Tabela 1)
### silver.tb_info_filmes (origem: tb_movies_info)
---

In [0]:
# Carregamos os dados da tabela Bronze
df_info = spark.table("workspace.bronze.tb_movies_info")

# Renomeamos as colunas
df_info = (
    df_info
    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("title", "titulo")
    .withColumnRenamed("original_title", "titulo_original")
    .withColumnRenamed("release_date", "data_lancamento")
    .withColumnRenamed("runtime", "duracao_minutos")
    .withColumnRenamed("original_language", "idioma_original")
    .withColumnRenamed("status", "status_filme")
    .withColumnRenamed("overview", "sinopse")
    .withColumnRenamed("tagline", "frase_divulgacao")
)

# Normalizar o status_filme:
# Remover espaços extras, substituir hífens por espaços e padronizar o texto em maiúsculas
df_info = df_info.withColumn(
    "status_normalizado",
    F.lower(
        F.trim(
            F.regexp_replace("status_filme", "-", " ")
        )
    )
)

# Traduzir os status conhecidos para português (valores que não puderem ser identificados serão classificados como "Não Informado").
df_info = df_info.withColumn(
    "status_filme",
    F.when(F.col("status_normalizado") == "released", "Lançado")
     .when(F.col("status_normalizado") == "post production", "Pós-Produção")
     .when(F.col("status_normalizado") == "in production", "Em Produção")
     .when(F.col("status_normalizado") == "planned", "Planejado")
     .when(F.col("status_normalizado") == "rumored", "Rumores")
     .when(F.col("status_normalizado") == "canceled", "Cancelado")
     .otherwise("Não Informado")
)

# Agora podemos remover a coluna auxiliar status_normalizado
df_info = df_info.drop("status_normalizado")

# Tentamos converter a data de lançamento utilizando diferentes formatos.
df_info = df_info.withColumn(
    "data_lancamento",
    F.coalesce(
        F.expr("try_to_date(data_lancamento, 'yyyy-MM-dd')"),
        F.expr("try_to_date(data_lancamento, 'MM-dd-yyyy')"),
        F.expr("try_to_date(data_lancamento, 'dd/MM/yyyy')"),
        F.expr("try_to_date(data_lancamento, 'MM/dd/yyyy')")
    )
)

# Conversão da duração_minutos para inteiro.
# Valores corrompidos ou deslocados na base são tratados como NULL.
df_info = df_info.withColumn(
    "duracao_minutos",
    F.expr("try_cast(duracao_minutos AS INT)")
)

# Criamos a coluna ano_lancamento a partir da data de lançamento.
df_info = df_info.withColumn(
    "ano_lancamento",
    F.year("data_lancamento")
)

# Para os registros duplicados do mesmo filme, devemos mantemos somente o registro com a ingestão mais recente
janela_filme = Window.partitionBy("id_filme").orderBy(
    F.col("ingestion_datetime").desc()
)
df_info = (
    df_info
    .withColumn("rn", F.row_number().over(janela_filme))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

# Selecionamos apenas as colunas finais da tabela Silver.
df_info = df_info.select(
    "id_filme",
    "titulo",
    "titulo_original",
    "data_lancamento",
    "ano_lancamento",
    "duracao_minutos",
    "idioma_original",
    "status_filme",
    "sinopse",
    "frase_divulgacao"
)

# Gravamos a tabela tratada na camada Silver.
(
    df_info.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.silver.tb_info_filmes")
)

display(df_info)


## Tabela 2)
### silver.tb_financeiro_filmes (origem: tb_movies_financials)
---

In [0]:
# Carregamos os dados da tabela Bronze
df_fin = spark.table("workspace.bronze.tb_movies_financials")

# Renomeamos as colunas
df_fin = (
    df_fin
    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("budget", "orcamento_usd")
    .withColumnRenamed("revenue", "receita_usd")
)

# Higienizar as colunas de orçamento e receita:
# remover símbolos, separadores e textos que representam ausência de valor.
df_fin = (
    df_fin
    .withColumn(
        "orcamento_usd",
        F.when(
            F.lower(F.trim(F.col("orcamento_usd"))).isin(
                "unknown", "não informado", ""
            ),
            None
        ).otherwise(
            F.regexp_replace(F.col("orcamento_usd"), r"[^0-9.-]", "")
        )
    )
    .withColumn(
        "receita_usd",
        F.when(
            F.lower(F.trim(F.col("receita_usd"))).isin(
                "unknown", "não informado", ""
            ),
            None
        ).otherwise(
            F.regexp_replace(F.col("receita_usd"), r"[^0-9.-]", "")
        )
    )
)

# Converter os valores para decimal.
df_fin = (
    df_fin
    .withColumn(
        "orcamento_usd",
        F.expr("try_cast(orcamento_usd AS DECIMAL(18,2))")
    )
    .withColumn(
        "receita_usd",
        F.expr("try_cast(receita_usd AS DECIMAL(18,2))")
    )
)

# Valores zerados ou negativos são tratados como ausentes
df_fin = (
    df_fin
    .withColumn(
        "orcamento_usd",
        F.when(F.col("orcamento_usd") > 0, F.col("orcamento_usd"))
    )
    .withColumn(
        "receita_usd",
        F.when(F.col("receita_usd") > 0, F.col("receita_usd"))
    )
)

display(df_fin)

In [0]:
# Agora iremos buscar a cotação mais recente disponível na tabela Bronze.
# Será utilizado apenas uma cotação para realizar a conversão dos valores para BRL.
df_cotacao_atual = (
    spark.table("workspace.bronze.tb_cotacao_dolar")
    .orderBy(F.col("dataHoraCotacao").desc())
    .limit(1)
    .select(
        F.col("cotacaoCompra")
        .cast("decimal(18,6)")
        .alias("cotacao_dolar")
    )
)

# Relacionamos agora a cotação a todos os filmes.
# Como df_cotacao_atual possui somente uma linha, o crossJoin
# apenas adiciona a mesma cotação a cada registro financeiro.
df_fin = df_fin.crossJoin(df_cotacao_atual)

# Calcular os valores equivalentes em (BRL)
df_fin = (
    df_fin
    .withColumn(
        "orcamento_brl",
        (F.col("orcamento_usd") * F.col("cotacao_dolar"))
        .cast("decimal(18,2)")
    )
    .withColumn(
        "receita_brl",
        (F.col("receita_usd") * F.col("cotacao_dolar"))
        .cast("decimal(18,2)")
    )
)

# Calcular o lucro em real.
# Caso orçamento ou receita estejam ausentes, o resultado correspondente permanecerá como NULL.
df_fin = (
    df_fin
    .withColumn(
        "lucro_usd",
        (F.col("receita_usd") - F.col("orcamento_usd"))
        .cast("decimal(18,2)")
    )
    .withColumn(
        "lucro_brl",
        (F.col("receita_brl") - F.col("orcamento_brl"))
        .cast("decimal(18,2)")
    )
)

# Cálculo da margem de lucro percentual.
# O cálculo somente é realizado quando o orçamento é válido e maior que zero
df_fin = df_fin.withColumn(
    "margem_lucro_percentual",
    F.when(
        F.col("orcamento_usd") > 0,
        (F.col("lucro_usd") / F.col("orcamento_usd")) * 100
    ).otherwise(None)
)

df_fin = df_fin.select(
    "id_filme",
    "orcamento_usd",
    "receita_usd",
    "orcamento_brl",
    "receita_brl",
    "lucro_usd",
    "lucro_brl",
    "margem_lucro_percentual"
)

# Removemos registros financeiros duplicados do mesmo filme.
df_fin = df_fin.dropDuplicates(["id_filme"])

display(df_fin)

df_fin.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.silver.tb_financeiro_filmes")

## Tabela 3)
### silver.tb_metricas_engajamento (origem: tb_movies_metrics)
---

In [0]:
# Carregar a tabela da camada Bronze
df_metricas = spark.table("workspace.bronze.tb_movies_metrics")

# Renomeamos as colunas conforme o mapeamento solicitado.
df_metricas = (
    df_metricas
    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("popularity", "popularidade")
    .withColumnRenamed("vote_average", "nota_media_tmdb")
    .withColumnRenamed("vote_count", "qtd_votos_tmdb")
    .withColumnRenamed("averageRating", "nota_media_imdb")
    .withColumnRenamed("numVotes", "qtd_votos_imdb")
)

# Padronizar o separador decimal da popularidade, substituindo vírgula por ponto antes da conversão numérica.
df_metricas = df_metricas.withColumn(
    "popularidade",
    F.regexp_replace(F.trim(F.col("popularidade")), ",", ".")
)

# Realizar agora a conversão numérica
df_metricas = (
    df_metricas
    .withColumn(
        "popularidade",
        F.expr("try_cast(popularidade AS DOUBLE)")
    )
    .withColumn(
        "nota_media_tmdb",
        F.expr("try_cast(nota_media_tmdb AS DOUBLE)")
    )
    .withColumn(
        "qtd_votos_tmdb",
        F.expr("try_cast(qtd_votos_tmdb AS INT)")
    )
    .withColumn(
        "nota_media_imdb",
        F.expr("try_cast(nota_media_imdb AS DOUBLE)")
    )
    .withColumn(
        "qtd_votos_imdb",
        F.expr("try_cast(qtd_votos_imdb AS INT)")
    )
)

# Notas médias fora do intervalo válido de 0 a 10 são consideradas inválidas e transformadas em NULL.
# Além de valores negativos de popularidade ou quantidade de votos.
df_metricas = (
    df_metricas
    .withColumn(
        "nota_media_tmdb",
        F.when(
            F.col("nota_media_tmdb").between(0, 10),
            F.col("nota_media_tmdb")
        )
    )
    .withColumn(
        "nota_media_imdb",
        F.when(
            F.col("nota_media_imdb").between(0, 10),
            F.col("nota_media_imdb")
        )
    )
    .withColumn(
        "popularidade",
        F.when(F.col("popularidade") >= 0, F.col("popularidade"))
    )
    .withColumn(
        "qtd_votos_tmdb",
        F.when(F.col("qtd_votos_tmdb") >= 0, F.col("qtd_votos_tmdb"))
    )
    .withColumn(
        "qtd_votos_imdb",
        F.when(F.col("qtd_votos_imdb") >= 0, F.col("qtd_votos_imdb"))
    )
)

# Removemos registros duplicados de métricas do mesmo filme.
df_metricas = df_metricas.dropDuplicates(["id_filme"])

display(df_metricas)

df_metricas = df_metricas.select(
    "id_filme",
    "popularidade",
    "nota_media_tmdb",
    "qtd_votos_tmdb",
    "nota_media_imdb",
    "qtd_votos_imdb"
)

df_metricas.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.silver.tb_metricas_engajamento")

## Tabela 4)
### silver.tb_avaliacoes_usuarios (origem: tb_movies_reviews)
---

In [0]:
# Carregar tabela da camada Bronze
df_avaliacoes = spark.table("workspace.bronze.tb_movies_reviews")

# Renomear as colunas
df_avaliacoes = (
    df_avaliacoes
    .withColumnRenamed("id", "id_filme")
    .withColumnRenamed("nome", "nome_usuario")
    .withColumnRenamed("nota", "nota_usuario")
    .withColumnRenamed("comentario", "comentario_usuario")
)

# Converter a nota para tipo numérico
df_avaliacoes = df_avaliacoes.withColumn(
    "nota_usuario",
    F.expr("try_cast(nota_usuario AS DOUBLE)")
)

# Notas fora do intervalo de 0 a 10 são consideradas inválidas
df_avaliacoes = df_avaliacoes.withColumn(
    "nota_usuario",
    F.when(
        F.col("nota_usuario").between(0, 10),
        F.col("nota_usuario")
    )
)

# Comentários vazios ou contendo apenas espaços são substituídos pelo texto padrão "Sem comentário"
df_avaliacoes = df_avaliacoes.withColumn(
    "comentario_usuario",
    F.when(
        F.trim(F.col("comentario_usuario")) == "",
        "Sem comentário"
    ).otherwise(
        F.col("comentario_usuario")
    )
)

# Tratamento de valores nulos de comentário
df_avaliacoes = df_avaliacoes.withColumn(
    "comentario_usuario",
    F.coalesce(
        F.col("comentario_usuario"),
        F.lit("Sem comentário")
    )
)

# Remover registros duplicados, considerando filme, usuário, nota e comentário.
df_avaliacoes = df_avaliacoes.dropDuplicates([
    "id_filme",
    "nome_usuario",
    "nota_usuario",
    "comentario_usuario"
])

df_avaliacoes = df_avaliacoes.select(
    "id_filme",
    "nome_usuario",
    "nota_usuario",
    "comentario_usuario"
)

display(df_avaliacoes)

df_avaliacoes.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.silver.tb_avaliacoes_usuarios")

## Tabela 5)
### silver.tb_generos (origem: tb_credits_and_tags, coluna genres)
---

In [0]:
# Carregar a tabela da camada Bronze
df_generos = spark.table("workspace.bronze.tb_credits_and_tags")

# Selecionar apenas o identificador do filme e a coluna de gêneros
df_generos = df_generos.select(
    F.col("id").alias("id_filme"),
    F.col("genres")
)

# Padronizar os separadores
df_generos = df_generos.withColumn(
    "genres",
    F.regexp_replace(F.col("genres"), r"[;|]", ",")
)

# Separar os gêneros em uma lista e criar uma linha para cada gênero
df_generos = df_generos.withColumn(
    "nome_genero",
    F.explode(F.split(F.col("genres"), ","))
)

# Remover espaços extras e padronizar a primeira letra maiúscula
df_generos = df_generos.withColumn(
    "nome_genero",
    F.initcap(F.trim(F.col("nome_genero")))
)

# Remover valores vazios ou nulos e puramente numéricos
df_generos = df_generos.filter(
    F.col("nome_genero").isNotNull() &
    (F.col("nome_genero") != "") &
    (~F.col("nome_genero").rlike(r"^\d+(\.\d+)?$"))
)

df_generos = (
    df_generos
    .select("id_filme", "nome_genero")
    .dropDuplicates()
)

display(df_generos)

df_generos.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.silver.tb_generos")

## Tabela 6)
### silver.tb_pessoas_empresas (origem: tb_credits_and_tags, colunas cast, directors, writers, production_companies)
---

In [0]:
# Carregar a tabela da camada Bronze
df_entidades = spark.table("workspace.bronze.tb_credits_and_tags")

# Criar uma tabela para cada tipo de entidade, separando os valores que estão agrupados em uma única coluna.
df_atores = (
    df_entidades
    .select(
        F.col("id").alias("id_filme"),
        F.explode(F.split(F.regexp_replace(F.col("cast"), ";", ","), ",")).alias("nome_entidade")
    )
    .withColumn("tipo_entidade", F.lit("Ator"))
)

df_diretores = (
    df_entidades
    .select(
        F.col("id").alias("id_filme"),
        F.explode(F.split(F.regexp_replace(F.col("directors"), ";", ","), ",")).alias("nome_entidade")
    )
    .withColumn("tipo_entidade", F.lit("Diretor"))
)

df_roteiristas = (
    df_entidades
    .select(
        F.col("id").alias("id_filme"),
        F.explode(F.split(F.regexp_replace(F.col("writers"), ";", ","), ",")).alias("nome_entidade")
    )
    .withColumn("tipo_entidade", F.lit("Roteirista"))
)

df_produtoras = (
    df_entidades
    .select(
        F.col("id").alias("id_filme"),
        F.explode(F.split(F.regexp_replace(F.col("production_companies"), ";", ","), ",")).alias("nome_entidade")
    )
    .withColumn("tipo_entidade", F.lit("Produtora"))
)

# Unificando todas as entidades em um único DataFrame
df_pessoas_empresas = (
    df_atores
    .unionByName(df_diretores)
    .unionByName(df_roteiristas)
    .unionByName(df_produtoras)
)

# Remover espaços extras e padronizar a primeira letra maiúscula
df_pessoas_empresas = df_pessoas_empresas.withColumn(
    "nome_entidade",
    F.initcap(F.trim(F.col("nome_entidade")))
)

# Remover valores vazios, nulos e puramente numéricos
df_pessoas_empresas = df_pessoas_empresas.filter(
    F.col("nome_entidade").isNotNull() &
    (F.col("nome_entidade") != "") &
    (~F.lower(F.col("nome_entidade")).isin(
        "nenhum",
        "none",
        "unknown",
        "não informado"
    )) &
    (~F.col("nome_entidade").rlike(r"^\d+(\.\d+)?$"))
)

# Remover registros duplicados
df_pessoas_empresas = df_pessoas_empresas.dropDuplicates()

display(df_pessoas_empresas)

df_pessoas_empresas.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.silver.tb_pessoas_empresas")


## Tabela 7)
### silver.tb_cotacao_dolar (origem: tb_cotacao_dolar)
---

In [0]:
# Carregar a tabela da camada Bronze
df_cotacao = spark.table("workspace.bronze.tb_cotacao_dolar")

# Converter a data/hora da cotação para uma coluna do tipo DATE
df_cotacao = df_cotacao.withColumn(
    "data_cotacao",
    F.to_date("dataHoraCotacao")
)

# Manteremos apenas uma cotação por dia. Como a API pode retornar mais de um registro no mesmo dia,
# utilizamos a cotação mais recente daquele dia.
janela_dia = Window.partitionBy("data_cotacao").orderBy(
    F.col("dataHoraCotacao").desc()
)
df_cotacao = (
    df_cotacao
    .withColumn("rn", F.row_number().over(janela_dia))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

# Descobrir a menor e a maior data disponíveis na cotação
limites = df_cotacao.select(
    F.min("data_cotacao").alias("data_min"),
    F.max("data_cotacao").alias("data_max")
).first()

data_min = limites["data_min"]
data_max = limites["data_max"]

# Criar um calendário contínuo entre a menor e a maior data
df_calendario = (
    spark.range(1)
    .select(
        F.explode(
            F.sequence(
                F.lit(data_min),
                F.lit(data_max),
                F.expr("interval 1 day")
            )
        ).alias("data_cotacao")
    )
)

# Associar o calendário com as cotações
df_cotacao_silver = (
    df_calendario
    .join(
        df_cotacao.select(
            "data_cotacao",
            F.col("cotacaoCompra").cast("double").alias("cotacao_compra")
        ),
        on="data_cotacao",
        how="left"
    )
)

# Aplicação do Forward Fill, onde os dias sem cotação recebem o valor do último dia útil disponível.
# A janela não utiliza partitionBy porque a cotação representa uma única série temporal contínua.
janela_ffill = (
    Window
    .orderBy("data_cotacao")
    .rowsBetween(Window.unboundedPreceding, 0)
)

df_cotacao_silver = df_cotacao_silver.withColumn(
    "cotacao_compra",
    F.last(
        "cotacao_compra",
        ignorenulls=True
    ).over(janela_ffill)
)

display(df_cotacao_silver)

df_cotacao_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.silver.tb_cotacao_dolar")